In [91]:
import sys
sys.path.append("../") # go to parent dir

from custom_helpers_py.utilities import camel_to_snake_case
import pandas as pd
from os import listdir
from os.path import join
import json

In [92]:
COMPRESSED_FOLDER_PATH = join("../.outFiles/analysis/compressed")
POLITICIANS_FP = join(COMPRESSED_FOLDER_PATH, "politician.csv")

politicians_df = pd.read_csv(POLITICIANS_FP, encoding="utf-8")
politicians_df

,p_full_name,p_chamber,p_state,p_district,p_party
0,Ralph Lee Abraham,H,Louisiana,5,R
1,Alma S Adams,H,North Carolina,12,D
2,Robert B Aderholt,H,Alabama,4,R
3,Pete Aguilar,H,California,33,D
4,Lamar Alexander,S,Tennessee,NaN,R
...,...,...,...,...,...
958,David Young,H,Iowa,3,R
959,Don Young,H,Alaska,At Large,R
960,Todd Young,S,Indiana,NaN,R
961,Lee M Zeldin,H,New York,1,R


In [93]:
from difflib import SequenceMatcher

def is_str(to_test):
    return isinstance(to_test, str)

def get_similarity_ratio(a: str, b:str):
    return SequenceMatcher(None, a, b).ratio()

KNOWN_NAME_FIX_DICT = {
    "WILLIAM M CASSIDY": "Bill Cassidy",
    "PAUL RYAN": "Paul D Ryan",
    "NICK J II RAHALL": "Nick J Rahall",
    "EARL LEROY CARTER": "Earl L \"Buddy\" Carter",
    "NICHOLAS V TAYLOR": "Van Taylor",
    "FACS DUNN": "Neil P Dunn",
    "RODNEY LELAND BLUM": "Rod Blum",
    "TJ JOHN (TJ) COX": "TJ Cox",
    "ROBERT P CORKER JR": "Bob Corker",
    "RAFAEL E CRUZ": "Ted Cruz",
    "JOHN F REED": "Jack Reed",
    "ROBERT P CORKER JR": "Bob Corker",
    "RODNEY LELAND BLUM": "Rod Blum",
    "JOHN F REED": "Jack Reed",
    "JAMES E BANKS": "Jim Banks",
}

def fix_full_name_in_df(in_df: pd.DataFrame) -> pd.DataFrame:
    correct_list_df = politicians_df[["p_full_name"]].drop_duplicates(subset="p_full_name")
    correct_list_df["p_full_name_upper"] =  correct_list_df["p_full_name"].str.upper()

    correct_list = correct_list_df[["p_full_name", "p_full_name_upper"]].to_dict(orient="records")

    correct_dict = {}
    for obj in correct_list:
        lower = obj["p_full_name"]
        upper = obj["p_full_name_upper"]
        correct_dict[upper] = lower
    
    def fix_in_name(in_row: dict):
        in_str:str = in_row["p_full_name"]
        original_name = in_str

        if not is_str(in_str):
            return pd.NA

        in_name = in_str.upper()

        TITLE_LIST = [
            "MR",
            "MS",
            "MRS",
            "DR",
            "MD",
            "HONORABLE"
        ]
        for title in TITLE_LIST:
            to_test = title + "-"
            in_name = in_name.replace(to_test, "")

            to_test = "-" + title
            in_name = in_name.replace(to_test, "")

            to_test = " " +  title + " "
            in_name = in_name.replace(to_test, "")

            to_test = " " + title
            in_name = in_name.replace(to_test, "")
        
        in_name = in_name.replace(" HON ", " ")
        in_name = in_name.replace(".", "")
        in_name = in_name.replace("-", " ")

        # Start mapping
        known = KNOWN_NAME_FIX_DICT.get(in_name, None)
        if known:
            return pd.Series({"p_full_name": original_name, "fix_name": known, "fix_confidence": 1})

        correction = correct_dict.get(in_name, None)
        if correction:
            return pd.Series({"p_full_name": original_name,"fix_name": correction, "fix_confidence": 1})
        
        # Approximate
        best_ratio = 0
        best_match = None
        for c_obj in correct_list:
            fn, fn_upper = c_obj["p_full_name"], c_obj["p_full_name_upper"]

            sim_ratio = get_similarity_ratio(in_name, fn_upper)
            if sim_ratio > best_ratio:
                best_ratio = sim_ratio
                best_match = fn
        
        if best_ratio > 0.5:
            return pd.Series({"p_full_name": original_name,"fix_name": best_match, "fix_confidence": best_ratio})
        return pd.Series({"p_full_name": original_name, "fix_name": pd.NA, "fix_confidence": 0})
    
    fix_df = in_df[["p_full_name"]].drop_duplicates(subset="p_full_name")
    fix_df: pd.DataFrame = fix_df.apply(fix_in_name, axis=1)

    in_df = in_df.merge(fix_df, on="p_full_name", how="left")
    in_df = in_df.rename(columns={
        "p_full_name": "old_p_full_name",
        "fix_name": "p_full_name",
        "fix_confidence": "fix_name_confidence"
    })
    return in_df

def merge_with_politicians_df(in_df: pd.DataFrame):
    in_df = in_df.merge(politicians_df, on="p_full_name", how="left", suffixes=("_old", ""))
    in_df = in_df.drop(columns=[col for col in in_df.columns if col.endswith("_old")])
    return in_df

def read_compressed_csv(in_name: str):
   return pd.read_csv(join(COMPRESSED_FOLDER_PATH, in_name + ".csv"), encoding="utf-8")  

In [94]:
# Fix self house
self_house_df = read_compressed_csv("self_house") 

def fix_asset_desc(in_desc: str):
   if not is_str(in_desc):
      return pd.NA

   in_desc = in_desc.replace("FILING STATUS NEW", "")
   return in_desc

self_house_df["asset_desc"] = self_house_df["asset_desc"].apply(fix_asset_desc)


self_house_df  = fix_full_name_in_df(self_house_df)
self_house_df = merge_with_politicians_df(self_house_df)
self_house_df = self_house_df.fillna(pd.NA)
   
self_house_df


,asset_name,ticker,asset_type,tx_type,owner,tx_date,notif_date,amount,asset_desc,old_p_full_name,tx_year,doc_id,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,DECATUR ALA CITY BRD ED SPL TAX SCH WTS,<NA>,<NA>,P,JT,07/3/2014,07/3/2014,$15001 - $50000,,MR-MO BROOKS,2014,20000606,SELF_HOUSE,Mo Brooks,1.0,H,Alabama,5,R
1,ETOWAH CNTY ALA BRD ED CAP OUTLAY WTS,<NA>,<NA>,P,JT,04/11/2014,04/11/2014,$1001 - $15000,,MR-MO BROOKS,2014,20000606,SELF_HOUSE,Mo Brooks,1.0,H,Alabama,5,R
2,MOBILE COUNTY ALA BRD SCH COMMRSCAP OUTLAY WTS,<NA>,<NA>,P,JT,04/16/2014,04/16/2014,$1001 - $15000,,MR-MO BROOKS,2014,20000606,SELF_HOUSE,Mo Brooks,1.0,H,Alabama,5,R
3,MORGAN STANLEY CAP TR V GTD CAP SECS (MWO),MWO,<NA>,S,JT,05/5/2014,05/5/2014,$1001 - $15000,,MR-MO BROOKS,2014,20000606,SELF_HOUSE,Mo Brooks,1.0,H,Alabama,5,R
4,PHENIX CITY AL SCH WTS GENL OBLIG- AT,<NA>,<NA>,P,JT,03/28/2014,03/28/2014,$15001 - $50000,,MR-MO BROOKS,2014,20000606,SELF_HOUSE,Mo Brooks,1.0,H,Alabama,5,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40359,PRINCE GEORGES CNTY MD GO CONSOLIDATED 5.00 DU...,<NA>,GS,S,JT,02/01/2024,02/01/2024,$500001 - $1000000,,SUZAN-K DELBENE,2024,20024495,SELF_HOUSE,Suzan K DelBene,1.0,H,Washington,1,D
40360,BRISTOL-MYERS SQUIBB COMPANY (BMY) | ST | (BMY),BMY,ST,S,<NA>,01/09/2024,01/12/2024,$1001 - $15000,SUBHOLDING OF RICHARD R LARSEN IRA RICK LARSE...,RICK LARSEN,2024,20024339,SELF_HOUSE,Rick Larsen,1.0,H,Washington,2,D
40361,COLGATE-PALMOLIVE COMPANY (CL) | ST |,CL,ST,P,<NA>,01/09/2024,01/12/2024,$1001 - $15000,SUBHOLDING OF RICHARD R LARSEN IRA RICK LARSE...,RICK LARSEN,2024,20024339,SELF_HOUSE,Rick Larsen,1.0,H,Washington,2,D
40362,THE HERSHEY COMPANY (HSY) | ST |,HSY,ST,S,<NA>,01/09/2024,01/12/2024,$1001 - $15000,SUBHOLDING OF RICHARD R LARSEN IRA RICK LARSE...,RICK LARSEN,2024,20024339,SELF_HOUSE,Rick Larsen,1.0,H,Washington,2,D


In [95]:
low_fix_confidence_df = self_house_df[self_house_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df

,old_p_full_name,p_full_name,fix_name_confidence
6352,KENNETH-R BUCK,Ken Buck,0.727273
6657,TOM-THOMAS-JR GRAVES,Tom Graves,0.666667
10262,CHARLIE-JOSEPH CRIST,Charlie Crist,0.787879
18710,NEAL-PATRICK-MD DUNN,Neal P Dunn,0.785714
20280,GREG-FRANCIS MURPHY,Gregory F Murphy,0.742857
20372,DAVID-CHESTON ROUZER,David Rouzer,0.750000
21812,NICHOLAS-VAN TAYLOR,Van Taylor,0.689655
22217,ELIZABETH FLETCHER,Lizzie Fletcher,0.787879
22604,DONALD-STERNOFF-JR BEYER,Donald S Beyer,0.736842
22815,BRYAN-GEORGE STEIL,Bryan Steil,0.758621


In [96]:
# Fix self senate
self_senate_df = read_compressed_csv("self_senate") 

self_senate_df  = fix_full_name_in_df(self_senate_df)
self_senate_df = merge_with_politicians_df(self_senate_df)
self_senate_df = self_senate_df.fillna(pd.NA)
   
self_senate_df


,tx_date,owner,ticker,asset_name,asset_type,tx_type,amount,comments,old_p_full_name,tx_year,doc_id,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,11/11/2014,Spouse,MDLZ,"Mondelez International, Inc. (NASDAQ)",<NA>,Sale (Full),"$50,001 - $100,000",--,ROY BLUNT,2014,9ddbcc76-dc18-4775-a7d5-a1a7063c0ebd,SELF_SENATE,Roy Blunt,1.0,S,Missouri,<NA>,R
1,04/08/2014,Self,AMT,American Tower Corporation (NYSE),<NA>,Sale (Full),"$15,001 - $50,000",--,CORY-A BOOKER,2014,29d797a6-e3ff-4d76-9ee9-2e3840adb15b,SELF_SENATE,Cory A Booker,1.0,S,New Jersey,<NA>,D
2,04/08/2014,Self,NFLX,"Netflix, Inc. (NASDAQ)",<NA>,Sale (Full),"$15,001 - $50,000",--,CORY-A BOOKER,2014,29d797a6-e3ff-4d76-9ee9-2e3840adb15b,SELF_SENATE,Cory A Booker,1.0,S,New Jersey,<NA>,D
3,08/08/2014,Self,NKE,"Nike, Inc. (NYSE)",<NA>,Sale (Full),"$1,001 - $15,000",--,CORY-A BOOKER,2014,7abb2400-6528-4f7f-9739-248dfedc3ca2,SELF_SENATE,Cory A Booker,1.0,S,New Jersey,<NA>,D
4,08/08/2014,Self,IRM,Iron Mountain Inc. (NYSE),<NA>,Sale (Full),"$1,001 - $15,000",--,CORY-A BOOKER,2014,7abb2400-6528-4f7f-9739-248dfedc3ca2,SELF_SENATE,Cory A Booker,1.0,S,New Jersey,<NA>,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18276,12/14/2023,Spouse,SNOW,Snowflake Inc Cl A,Stock,Purchase,"$1,001 - $15,000",--,SHELDON WHITEHOUSE,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
18277,12/14/2023,Spouse,LLY,Eli Lilly and Company,Stock,Purchase,"$1,001 - $15,000",--,SHELDON WHITEHOUSE,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
18278,12/07/2023,Self,TGT,Target Corp,Stock,Sale (Full),"$15,001 - $50,000",--,SHELDON WHITEHOUSE,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
18279,12/07/2023,Self,KO,Coca-Cola Company,Stock,Purchase,"$1,001 - $15,000",--,SHELDON WHITEHOUSE,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D


In [97]:
low_fix_confidence_df = self_senate_df[self_senate_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df

,old_p_full_name,p_full_name,fix_name_confidence
1697,JEFFERSON-B SESSIONS-III,Jeff Sessions,0.702703
5305,JOSEPH MANCHIN-III,Joe Manchin,0.758621
6877,TIMOTHY-M KAINE,Tim Kaine,0.750000
8877,A-MITCHELL MCCONNELL-JR,Mitch McConnell,0.789474
15262,WILLIAM-F HAGERTY-IV,Bill Hagerty,0.687500
17318,JOHN-P RICKETTS,Pete Ricketts,0.714286


In [98]:
# Fix watcher house
watcher_house_df = read_compressed_csv("watcher_house") 

# Remove her because she's from puerto rico, not officially a member
watcher_house_df = watcher_house_df[watcher_house_df["p_full_name"] != "Ada Norah Henriquez"]
 
watcher_house_df  = fix_full_name_in_df(watcher_house_df)
watcher_house_df = merge_with_politicians_df(watcher_house_df)
watcher_house_df = watcher_house_df.fillna(pd.NA)
   
watcher_house_df

,notif_date,tx_date,owner,ticker,asset_name,tx_type,amount,old_p_full_name,cap_gains_over_200_usd,asset_industry,asset_sector,doc_id,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,10/04/2021,2021-09-27,joint,BP,BP plc,purchase,"$1,001 - $15,000",Virginia Foxx,False,Integrated oil Companies,Energy,20019557,WATCHER_HOUSE,Virginia Foxx,1.0,H,North Carolina,5,R
1,10/04/2021,2021-09-13,joint,XOM,Exxon Mobil Corporation,purchase,"$1,001 - $15,000",Virginia Foxx,False,Integrated oil Companies,Energy,20019557,WATCHER_HOUSE,Virginia Foxx,1.0,H,North Carolina,5,R
2,10/04/2021,2021-09-10,joint,ILPT,Industrial Logistics Properties Trust - Common...,purchase,"$15,001 - $50,000",Virginia Foxx,False,Real Estate Investment Trusts,Real Estate,20019557,WATCHER_HOUSE,Virginia Foxx,1.0,H,North Carolina,5,R
3,10/04/2021,2021-09-28,joint,PM,Phillip Morris International Inc,purchase,"$15,001 - $50,000",Virginia Foxx,False,Farming/Seeds/Milling,Consumer Non-Durables,20019557,WATCHER_HOUSE,Virginia Foxx,1.0,H,North Carolina,5,R
4,10/04/2021,2021-09-17,self,BLK,BlackRock Inc,sale_partial,"$1,001 - $15,000",Alan S. Lowenthal,False,Investment Bankers/Brokers/Service,Finance,20019570,WATCHER_HOUSE,Alan S Lowenthal,1.0,H,California,47,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17161,06/10/2020,2020-04-09,--,SWK,"Stanley Black & Decker, Inc.",sale_partial,"$1,001 - $15,000",Ed Perlmutter,False,Diversified Manufacture,Consumer Discretionary,20016738,WATCHER_HOUSE,Ed Perlmutter,1.0,H,Colorado,7,D
17162,06/10/2020,2020-04-09,--,USB,U.S. Bancorp,sale_partial,"$1,001 - $15,000",Ed Perlmutter,False,Major Banks,Finance,20016738,WATCHER_HOUSE,Ed Perlmutter,1.0,H,Colorado,7,D
17163,06/10/2020,2020-03-13,<NA>,BMY,Bristol-Myers Squibb Company,sale_full,"$100,001 - $250,000",Van Taylor,False,Major Pharmaceuticals,Health Care,20016703,WATCHER_HOUSE,Van Taylor,1.0,H,Texas,3,R
17164,06/10/2020,2020-03-13,<NA>,LLY,Eli Lilly and Company,sale_full,"$500,001 - $1,000,000",Van Taylor,False,Major Pharmaceuticals,Health Care,20016703,WATCHER_HOUSE,Van Taylor,1.0,H,Texas,3,R


In [99]:
low_fix_confidence_df = watcher_house_df[watcher_house_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df

,old_p_full_name,p_full_name,fix_name_confidence
895,Michael Patrick Guest,Michael Guest,0.764706
1404,Harold Dallas Rogers,Harold Rogers,0.787879
2987,David Cheston Rouzer,David Rouzer,0.750000
6617,Greg Francis Murphy,Gregory F Murphy,0.742857
6753,Ashley Hinson Arenholz,Ashley Hinson,0.742857
10724,James M. Costa,Jim Costa,0.727273
12800,Felix Barry Moore,Barry Moore,0.785714
13126,Dan Daniel Bishop,Dan Bishop,0.740741


In [100]:
# Fix watcher senate
watcher_senate_df = read_compressed_csv("watcher_senate") 

watcher_senate_df  = fix_full_name_in_df(watcher_senate_df)
watcher_senate_df = merge_with_politicians_df(watcher_senate_df)
watcher_senate_df = watcher_senate_df.fillna(pd.NA)
   
watcher_senate_df

,tx_date,owner,ticker,asset_name,asset_type,tx_type,amount,comments,asset_industry,asset_sector,old_p_full_name,notif_date,doc_id,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,04/18/2023,Spouse,ESS,"Essex Property Trust, Inc. Common Stock",Stock,Sale (Full),"$1,001 - $15,000",--,Real Estate Investment Trusts,Consumer Services,Sheldon Whitehouse,05/17/2023,9fc025a0-f893-47b2-9252-a2820737a409,WATCHER_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
1,04/18/2023,Self,ESS,"Essex Property Trust, Inc. Common Stock",Stock,Sale (Full),"$1,001 - $15,000",--,Real Estate Investment Trusts,Consumer Services,Sheldon Whitehouse,05/17/2023,9fc025a0-f893-47b2-9252-a2820737a409,WATCHER_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
2,05/16/2023,<NA>,<NA>,This filing was disclosed via scanned PDF. Use...,PDF Disclosed Filing,<NA>,Unknown,<NA>,<NA>,<NA>,Michael F. Bennet,05/16/2023,f590a331-1f74-4d08-b79f-88930593f314,WATCHER_SENATE,Michael F Bennet,1.0,S,Colorado,<NA>,D
3,04/04/2023,Spouse,UPS,"United Parcel Service, Inc. Common Stock",Stock,Sale (Full),"$1,001 - $15,000",--,Trucking Freight/Courier Services,Transportation,Shelley Moore Capito,05/15/2023,60987939-a116-41ee-86c2-8ba920461691,WATCHER_SENATE,Shelley Moore Capito,1.0,S,West Virginia,<NA>,R
4,04/04/2023,Spouse,MCD,McDonald's Corporation Common Stock,Stock,Sale (Partial),"$1,001 - $15,000",--,Restaurants,Consumer Services,Shelley Moore Capito,05/15/2023,60987939-a116-41ee-86c2-8ba920461691,WATCHER_SENATE,Shelley Moore Capito,1.0,S,West Virginia,<NA>,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8441,08/17/2012,<NA>,<NA>,This filing was disclosed via scanned PDF. Use...,PDF Disclosed Filing,<NA>,Unknown,<NA>,<NA>,<NA>,Sheldon Whitehouse,08/17/2012,5221D7D8-15D0-40FE-A64C-DD242311F0AE,WATCHER_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
8442,08/16/2012,<NA>,<NA>,This filing was disclosed via scanned PDF. Use...,PDF Disclosed Filing,<NA>,Unknown,<NA>,<NA>,<NA>,Pat Roberts,08/16/2012,5C666F29-7055-461D-B4F7-5EA73AFCD860,WATCHER_SENATE,Pat Roberts,1.0,S,Kansas,<NA>,R
8443,08/15/2012,<NA>,<NA>,This filing was disclosed via scanned PDF. Use...,PDF Disclosed Filing,<NA>,Unknown,<NA>,<NA>,<NA>,Rob Portman,08/15/2012,0D78FC31-D28A-440A-8E2D-65C4D9861EAB,WATCHER_SENATE,Rob Portman,1.0,S,Ohio,<NA>,R
8444,08/02/2012,<NA>,<NA>,This filing was disclosed via scanned PDF. Use...,PDF Disclosed Filing,<NA>,Unknown,<NA>,<NA>,<NA>,Thomas R. Carper,08/02/2012,CFDE3B80-E8BD-4F2D-9D64-E892C5EFB32A,WATCHER_SENATE,Thomas R Carper,1.0,S,Delaware,<NA>,D


In [101]:
low_fix_confidence_df = watcher_senate_df[watcher_senate_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df

,old_p_full_name,p_full_name,fix_name_confidence


In [102]:
# Fix trendspider
trendspider_df = read_compressed_csv("trendspider") 

def fix_name(in_name: str):
    if "," in in_name:
        tmp = in_name.split(",")
        in_name = tmp[1] + " " + tmp[0]
    in_name = in_name.strip()
    return in_name

trendspider_df["p_full_name"] = trendspider_df["p_full_name"].apply(fix_name)
trendspider_df  = fix_full_name_in_df(trendspider_df)
trendspider_df = merge_with_politicians_df(trendspider_df)
trendspider_df = trendspider_df.fillna(pd.NA)
   
trendspider_df

,ticker,asset_name,old_p_full_name,tx_type,amount,tx_date,notif_date,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,NGL,Ngl Energy Partners Lp Common Units Representi...,Mark Dr Green,Sale,"$100,001 - $250,000","Apr 1, 2024","Apr 7, 2024",TREND_SPIDER,Mark E Green,0.857143,H,Tennessee,7,R
1,XNGSY,Een Energy Hldgs Unsp/Adr,Josh Gottheimer,Purchase,"$1,001 - $15,000","Mar 26, 2024","Apr 7, 2024",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
2,AAPL,Apple Inc. - Common Stock,Josh Gottheimer,Sale,"$1,001 - $15,000","Mar 22, 2024","Apr 7, 2024",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
3,SQ,"Block, Inc. Class A Common Stock,",Josh Gottheimer,Purchase,"$1,001 - $15,000","Mar 21, 2024","Apr 7, 2024",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
4,AMD,"Advanced Micro Devices, Inc.",Josh Gottheimer,Purchase,"$1,001 - $15,000","Mar 18, 2024","Apr 7, 2024",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52178,HCA,"Hca Healthcare, Inc.",Josh Gottheimer,Sale,"$1,001 - $15,000","Sep 28, 2022","Oct 16, 2022",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
52179,DXCM,"Dexcom, Inc.",Josh Gottheimer,Sale,"$1,001 - $15,000","Sep 28, 2022","Oct 16, 2022",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
52180,ORCL,Oracle Corporation,Josh Gottheimer,Sale,"$1,001 - $15,000","Sep 28, 2022","Oct 16, 2022",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
52181,ALNY,"Alnylam Pharmaceuticals, Inc.",Josh Gottheimer,Purchase,"$1,001 - $15,000","Sep 28, 2022","Oct 16, 2022",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D


In [103]:
low_fix_confidence_df = trendspider_df[trendspider_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df


,old_p_full_name,p_full_name,fix_name_confidence
65,A. Mitchell Jr. McConnell,Mitch McConnell,0.789474
85,Michael Patrick Guest,Michael Guest,0.764706
356,Cindy Axne,Cynthia Axne,0.636364
2385,Donald Sternoff Beyer Jr.,Donald S Beyer,0.736842
3336,David Cheston Rouzer,David Rouzer,0.750000
4209,Felix Barry Moore,Barry Moore,0.785714
4243,Kenneth R. Buck,Ken Buck,0.727273
8409,Michael John Gallagher,Mike Gallagher,0.722222
8418,Ashley Hinson Arenholz,Ashley Hinson,0.742857
13408,James Calhoun,James E Clyburn,0.714286
